# Model Evaluation: ROUGE, BLEU, and BERTScore

This notebook evaluates two LLM approaches on **Austrian law** questions from `dataset_clean.csv`. Generated answers are compared against human-written answers using automated metrics.

**Models evaluated:**
- **Model 1 — Gemma 4 31B** (API inference, zero-shot): A large 31B-parameter model accessed via Google AI Studio API with expert system prompting.
- **Model 2 — Gemma 4 E2B** (QLoRA fine-tuned): A smaller 2.3B effective (5.1B with embeddings) model fine-tuned on German law Q&A data using QLoRA (4-bit quantization + LoRA adapters).

**Metrics used:**
- **ROUGE** (Recall-Oriented Understudy for Gisting Evaluation): Measures n-gram overlap between generated and reference answers. ROUGE-1 checks unigrams, ROUGE-2 checks bigrams, ROUGE-L checks longest common subsequence.
- **BLEU** (Bilingual Evaluation Understudy): Measures precision of n-gram matches. Originally designed for machine translation, it penalizes outputs that are too short (brevity penalty).
- **BERTScore**: Uses contextual embeddings from a pre-trained BERT model to compute semantic similarity. More robust to paraphrasing than surface-level metrics like ROUGE/BLEU.

## 1. Setup and Installation

In [1]:
!pip install --upgrade pip
# Install evaluation libraries
# rouge-score: Google's ROUGE implementation
# nltk: tokenization for BLEU
# bert-score: contextual embedding-based evaluation (requires PyTorch)
!pip install rouge-score nltk bert-score pandas -q

import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/sipan34/VS Code/python_coding/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/sipan34/VS Code/python_coding/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/sipan34/VS Code/python_coding/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", lin

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score_fn
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("All libraries imported successfully.")

All libraries imported successfully.


## 2. Load and Prepare Data

Three CSV files are loaded:
- `ground_truth.csv` — the human-written answers (643 questions with correct answers and legal source references)
- `model1_api_inference_results.csv` — answers from Gemma 4 31B (API inference)
- `model2_finetune_results.csv` — answers from Gemma 4 E2B (QLoRA fine-tuned)

Both model outputs contain `<end_of_turn>` tokens from the Gemma tokenizer that need to be stripped before evaluation.

In [3]:
# Load all three CSV files
ans_df = pd.read_csv("../results/dataset_answers.csv")
m1_df = pd.read_csv("../results/model1_api_inference_results.csv")
m2_df = pd.read_csv("../results/model2_finetune_results.csv")

In [4]:
# Clean the <end_of_turn> tokens that Gemma appends to its outputs
# These are generation artifacts and would unfairly affect ROUGE/BLEU scores
m1_df['answer'] = m1_df['answer'].fillna("").astype(str).str.replace('<end_of_turn>', '', regex=False).str.strip()
m2_df['answer'] = m2_df['answer'].fillna("").astype(str).str.replace('<end_of_turn>', '', regex=False).str.strip()
ans_df['correct_answer'] = ans_df['correct_answer'].fillna("").astype(str).str.replace('<end_of_turn>', '', regex=False).str.strip()

# Merge model outputs with ground truth on the question ID
# This ensures only questions that have both a prediction and a reference are evaluated
merged_m1 = pd.merge(ans_df, m1_df, on='id', suffixes=('_gt', '_pred'))
merged_m2 = pd.merge(ans_df, m2_df, on='id', suffixes=('_gt', '_pred'))

print(f"Model 1: {len(merged_m1)} matched question-answer pairs")
print(f"Model 2: {len(merged_m2)} matched question-answer pairs")

Model 1: 643 matched question-answer pairs
Model 2: 643 matched question-answer pairs


## 3. Evaluation: ROUGE Scores

ROUGE measures recall-oriented overlap between the generated answer and the reference. Three variants are computed:
- **ROUGE-1**: Unigram overlap — how many individual words from the reference appear in the prediction.
- **ROUGE-2**: Bigram overlap — how many consecutive word pairs match. Captures some phrase-level similarity.
- **ROUGE-L**: Longest Common Subsequence — measures the longest matching sequence of words in order, even if not contiguous.

All scores are F1 (harmonic mean of precision and recall), averaged across all 643 questions.

In [ ]:
# Initialize ROUGE scorer
# use_stemmer=False because German stemming is not well-supported in the default scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

# Compute ROUGE scores for both models
# Loop through each question, score each prediction against the reference, store results
rouge_m1 = {'rouge1': [], 'rouge2': [], 'rougeL': []}
rouge_m2 = {'rouge1': [], 'rouge2': [], 'rougeL': []}

for _, row in merged_m1.iterrows():
    result = scorer.score(str(row['correct_answer']), str(row['answer']))
    # .fmeasure is the F1 score — balances precision and recall
    rouge_m1['rouge1'].append(result['rouge1'].fmeasure) 
    rouge_m1['rouge2'].append(result['rouge2'].fmeasure)
    rouge_m1['rougeL'].append(result['rougeL'].fmeasure)

for _, row in merged_m2.iterrows():
    result = scorer.score(str(row['correct_answer']), str(row['answer']))
    rouge_m2['rouge1'].append(result['rouge1'].fmeasure)
    rouge_m2['rouge2'].append(result['rouge2'].fmeasure)
    rouge_m2['rougeL'].append(result['rougeL'].fmeasure)

# Store per-question scores for later error analysis
merged_m1['rouge1'] = rouge_m1['rouge1']
merged_m1['rouge2'] = rouge_m1['rouge2']
merged_m1['rougeL'] = rouge_m1['rougeL']

merged_m2['rouge1'] = rouge_m2['rouge1']
merged_m2['rouge2'] = rouge_m2['rouge2']
merged_m2['rougeL'] = rouge_m2['rougeL']

# Print average scores
print(f"{'Metric':<12} {'Model 1 (31B)':<18} {'Model 2 (E2B-FT)':<18}")
print("-" * 48)
for metric in ['rouge1', 'rouge2', 'rougeL']:
    avg1 = np.mean(rouge_m1[metric])
    avg2 = np.mean(rouge_m2[metric])
    label = metric.upper().replace('ROUGE', 'ROUGE-')
    print(f"{label:<12} {avg1:<18.4f} {avg2:<18.4f}")

Metric       Model 1 (31B)      Model 2 (E2B-FT)  
------------------------------------------------
ROUGE-1      0.1816             0.2167            
ROUGE-2      0.0571             0.0643            
ROUGE-L      0.1152             0.1572            


## 4. Evaluation: BLEU Scores

BLEU (Bilingual Evaluation Understudy) is a precision-based metric originally designed for machine translation. It measures how many n-grams in the generated text appear in the reference.

Key properties:
- **Precision-oriented**: counts how many predicted n-grams match the reference (opposite of ROUGE's recall focus).
- **Brevity penalty**: penalizes predictions that are much shorter than the reference.
- **Smoothing**: Method 1 smoothing is applied to handle cases where higher-order n-grams have zero matches (common for short answers).

Both reference and prediction are tokenized using NLTK's word tokenizer, and sentence-level BLEU is computed with 1-to-4-gram weights.

In [6]:
# Smoothing function to avoid zero scores when higher n-grams don't match
# Method 1 adds a small epsilon to zero counts
smooth_fn = SmoothingFunction().method1

# Compute BLEU for Model 1
bleu_m1 = []
for _, row in merged_m1.iterrows():
    # Tokenize both reference and prediction (lowercased for fair comparison)
    ref_tokens = nltk.word_tokenize(str(row['correct_answer']).lower())
    pred_tokens = nltk.word_tokenize(str(row['answer']).lower())
    # sentence_bleu expects reference as a list of token lists (supports multiple refs)
    score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth_fn) if pred_tokens else 0.0
    bleu_m1.append(score)

# Compute BLEU for Model 2
bleu_m2 = []
for _, row in merged_m2.iterrows():
    ref_tokens = nltk.word_tokenize(str(row['correct_answer']).lower())
    pred_tokens = nltk.word_tokenize(str(row['answer']).lower())
    score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth_fn) if pred_tokens else 0.0
    bleu_m2.append(score)

# Store per-question BLEU scores
merged_m1['bleu'] = bleu_m1
merged_m2['bleu'] = bleu_m2

print(f"Average BLEU:")
print(f"  Model 1 (Gemma 4 31B):        {np.mean(bleu_m1):.4f}")
print(f"  Model 2 (Gemma 4 E2B FT):     {np.mean(bleu_m2):.4f}")

Average BLEU:
  Model 1 (Gemma 4 31B):        0.0208
  Model 2 (Gemma 4 E2B FT):     0.0307


## 5. Evaluation: BERTScore

BERTScore uses pre-trained contextual embeddings to compute token-level similarity between the reference and the prediction. Unlike ROUGE and BLEU, BERTScore captures **semantic similarity** — two sentences can score high even if they use different words to express the same meaning.

The `bert-base-multilingual-cased` model is used here since the data is in German.

In [7]:
# Compute BERTScore for Model 1
print("Computing BERTScore for Model 1...")
P1, R1, F1_m1 = bert_score_fn(
    merged_m1['answer'].tolist(),
    merged_m1['correct_answer'].tolist(),
    model_type="bert-base-multilingual-cased",
    lang="de", #German
    verbose=True
)

# Compute BERTScore for Model 2
print("\nComputing BERTScore for Model 2...")
P2, R2, F1_m2 = bert_score_fn(
    merged_m2['answer'].tolist(),
    merged_m2['correct_answer'].tolist(),
    model_type="bert-base-multilingual-cased",
    lang="de",
    verbose=True
)

# Store per-question BERTScore F1
merged_m1['bertscore_f1'] = [f.item() for f in F1_m1]
merged_m2['bertscore_f1'] = [f.item() for f in F1_m2]

print(f"\nAverage BERTScore F1:")
print(f"  Model 1 (Gemma 4 31B):    {F1_m1.mean():.4f}")
print(f"  Model 2 (Gemma 4 E2B FT): {F1_m2.mean():.4f}")

Computing BERTScore for Model 1...
calculating scores...
computing bert embedding.


100%|██████████| 19/19 [13:19<00:00, 42.07s/it]


computing greedy matching.


100%|██████████| 11/11 [00:05<00:00,  1.85it/s]


done in 805.91 seconds, 0.80 sentences/sec

Computing BERTScore for Model 2...
calculating scores...
computing bert embedding.


100%|██████████| 19/19 [04:24<00:00, 13.90s/it]


computing greedy matching.


100%|██████████| 11/11 [00:01<00:00,  5.50it/s]


done in 266.38 seconds, 2.41 sentences/sec

Average BERTScore F1:
  Model 1 (Gemma 4 31B):    0.6891
  Model 2 (Gemma 4 E2B FT): 0.7113


## 6. Main Results Table

Summary of all evaluation metrics. Models as rows, metrics as columns. Higher is better for all metrics.

In [8]:
# Build the results summary table
results_table = pd.DataFrame({
    'Model': ['Gemma 4 31B (Inference)', 'Gemma 4 E2B (QLoRA Fine-tuned)'],
    'ROUGE-1': [np.mean(rouge_m1['rouge1']), np.mean(rouge_m2['rouge1'])],
    'ROUGE-2': [np.mean(rouge_m1['rouge2']), np.mean(rouge_m2['rouge2'])],
    'ROUGE-L': [np.mean(rouge_m1['rougeL']), np.mean(rouge_m2['rougeL'])],
    'BLEU': [np.mean(bleu_m1), np.mean(bleu_m2)],
    'BERTScore F1': [F1_m1.mean().item(), F1_m2.mean().item()]
})

# Format numeric columns to 4 decimal places
numeric_cols = [c for c in results_table.columns if c != 'Model']
results_table[numeric_cols] = results_table[numeric_cols].round(4)


print("MAIN RESULTS TABLE")

print(results_table.to_string(index=False))
print("\nAll scores are F1 (harmonic mean of precision and recall). Higher is better.")

MAIN RESULTS TABLE
                         Model  ROUGE-1  ROUGE-2  ROUGE-L   BLEU  BERTScore F1
       Gemma 4 31B (Inference)   0.1816   0.0571   0.1152 0.0208        0.6891
Gemma 4 E2B (QLoRA Fine-tuned)   0.2167   0.0643   0.1572 0.0307        0.7113

All scores are F1 (harmonic mean of precision and recall). Higher is better.


## 7. Error Analysis (Bonus)

The main failure modes of each model are investigated by examining:
1. **Score distribution** — how are ROUGE-L scores spread across questions?
2. **Answer length** — does the model over-generate or under-generate?
3. **Hallucinated legal references** — does the model cite incorrect or non-existent law paragraphs?
4. **Repetition** — does the model get stuck in loops?
5. **Worst examples** — concrete examples of the lowest-scoring answers.

In [9]:
# 7a. Score Distribution
# How many answers fall into each quality bucket?
print("7a. ROUGE-L SCORE DISTRIBUTION")

for name, df in [("Model 1 (31B)", merged_m1), ("Model 2 (E2B-FT)", merged_m2)]:
    zero  = (df['rougeL'] == 0).sum()
    low   = ((df['rougeL'] > 0) & (df['rougeL'] < 0.1)).sum()
    med   = ((df['rougeL'] >= 0.1) & (df['rougeL'] < 0.3)).sum()
    high  = (df['rougeL'] >= 0.3).sum()
    total = len(df)
    
    print(f"\n{name}:")
    print(f"  Zero (0.0):         {zero:>4} ({100*zero/total:.1f}%)")
    print(f"  Low (0.01-0.10):    {low:>4} ({100*low/total:.1f}%)")
    print(f"  Medium (0.10-0.30): {med:>4} ({100*med/total:.1f}%)")
    print(f"  High (>0.30):       {high:>4} ({100*high/total:.1f}%)")

7a. ROUGE-L SCORE DISTRIBUTION

Model 1 (31B):
  Zero (0.0):            0 (0.0%)
  Low (0.01-0.10):     290 (45.1%)
  Medium (0.10-0.30):  349 (54.3%)
  High (>0.30):          4 (0.6%)

Model 2 (E2B-FT):
  Zero (0.0):           16 (2.5%)
  Low (0.01-0.10):     127 (19.8%)
  Medium (0.10-0.30):  464 (72.2%)
  High (>0.30):         36 (5.6%)


In [ ]:
# 7b. Answer Length Analysis
# Overly long answers dilute ROUGE precision; overly short answers miss recall
print("7b. ANSWER LENGTH ANALYSIS")


gt_len  = merged_m1['correct_answer'].str.len().mean()
m1_len  = merged_m1['answer'].str.len().mean()
m2_len  = merged_m2['answer'].str.len().mean()

print(f"\nAverage answer length (characters):")
print(f"  Ground truth:   {gt_len:>7.0f}")
print(f"  Model 1 (31B):  {m1_len:>7.0f}  ({m1_len/gt_len:.1f}x reference length)")
print(f"  Model 2 (E2B):  {m2_len:>7.0f}  ({m2_len/gt_len:.1f}x reference length)")

print(f"\nModel 1 generates answers that are ~{m1_len/gt_len:.1f}x longer than the reference.")
print(f"This verbosity hurts ROUGE precision since many generated words don't match the reference.")
print(f"Model 2 matches the reference length much more closely after fine-tuning.")

7b. ANSWER LENGTH ANALYSIS

Average answer length (characters):
  Ground truth:       279
  Model 1 (31B):     1310  (4.7x reference length)
  Model 2 (E2B):      282  (1.0x reference length)

Model 1 generates answers that are ~4.7x longer than the reference.
This verbosity hurts ROUGE precision since many generated words don't match the reference.
Model 2 matches the reference length much more closely after fine-tuning.


In [11]:
# 7c. Hallucinated Legal References
# The dataset is about Austrian tax law (KStG, EStG, UStG, BAO).
# Citing BGB (German civil code) or GmbHG in tax law context is a hallucination.
print("7c. HALLUCINATED LEGAL REFERENCES")


for name, df in [("Model 1 (31B)", merged_m1), ("Model 2 (E2B-FT)", merged_m2)]:
    kstg  = df['answer'].str.contains('KStG', na=False).sum()
    estg  = df['answer'].str.contains('EStG', na=False).sum()
    bgb   = df['answer'].str.contains('BGB', na=False).sum()
    gmbhg = df['answer'].str.contains('GmbHG', na=False).sum()
    
    print(f"\n{name} — law references in {len(df)} answers:")
    print(f"  KStG (correct):      {kstg:>4}")
    print(f"  EStG (correct):      {estg:>4}")
    print(f"  BGB (hallucinated):  {bgb:>4}")
    print(f"  GmbHG (hallucinated):{gmbhg:>4}")

print(f"\nModel 2 hallucinates BGB references significantly more often than Model 1.")
print(f"This suggests the German law fine-tuning dataset included general civil law,")
print(f"causing the model to confuse tax law with civil law concepts.")

7c. HALLUCINATED LEGAL REFERENCES

Model 1 (31B) — law references in 643 answers:
  KStG (correct):       118
  EStG (correct):       445
  BGB (hallucinated):    10
  GmbHG (hallucinated):   2

Model 2 (E2B-FT) — law references in 643 answers:
  KStG (correct):       287
  EStG (correct):       177
  BGB (hallucinated):    74
  GmbHG (hallucinated):   4

Model 2 hallucinates BGB references significantly more often than Model 1.
This suggests the German law fine-tuning dataset included general civil law,
causing the model to confuse tax law with civil law concepts.


In [12]:
# 7d. Repetition Detection
# Some models get stuck in loops, repeating the same sentence multiple times
# Here each answer is split into sentences and checked for exact duplicates
print("7d. REPETITION DETECTION")

for name, df in [("Model 1 (31B)", merged_m1), ("Model 2 (E2B-FT)", merged_m2)]:
    rep_count = 0
    for _, row in df.iterrows():
        sentences = str(row['answer']).split('.')
        # Check if any sentence of substantial length (>20 chars) appears more than once
        seen = set()
        has_dup = False
        for s in sentences:
            s = s.strip()
            if len(s) > 20:
                if s in seen:
                    has_dup = True
                    break
                seen.add(s)
        if has_dup:
            rep_count += 1
    print(f"\n{name}: {rep_count} repetitive answers ({100*rep_count/len(df):.1f}%)")

print(f"\nModel 2 shows some repetition issues (~4%), likely due to the small model size")
print(f"and limited fine-tuning (only 150 steps). Model 1 does not repeat, benefiting")
print(f"from its larger capacity and the API's built-in generation safeguards.")

7d. REPETITION DETECTION

Model 1 (31B): 0 repetitive answers (0.0%)

Model 2 (E2B-FT): 25 repetitive answers (3.9%)

Model 2 shows some repetition issues (~4%), likely due to the small model size
and limited fine-tuning (only 150 steps). Model 1 does not repeat, benefiting
from its larger capacity and the API's built-in generation safeguards.


In [13]:
# 7e. Worst Predictions — Concrete Error Examples
# The 5 lowest-scoring answers for each model show the main failure modes
print("7e. WORST PREDICTIONS (5 Lowest ROUGE-L)")

for name, df in [("Model 1 (31B)", merged_m1), ("Model 2 (E2B-FT)", merged_m2)]:
    print(f"\n--- {name} ---")
    worst = df.nsmallest(5, 'rougeL')
    for _, row in worst.iterrows():
        print(f"\n  ID: {row['id']}  |  ROUGE-L: {row['rougeL']:.4f}")
        print(f"  Reference: {str(row['correct_answer'])[:200]}")
        print(f"  Predicted: {str(row['answer'])[:200]}")

7e. WORST PREDICTIONS (5 Lowest ROUGE-L)

--- Model 1 (31B) ---

  ID: SELF-042  |  ROUGE-L: 0.0084
  Reference: Durch Betriebsvermögensvergleich (Bilanzierung).
  Predicted: Der Gewinn aus Gewerbebetrieb und selbstständiger Arbeit wird gemäß § 4 Abs. 1 EStG grundsätzlich auf Basis einer Bilanz und einer Gewinn- und Verlustrechnung (GuV) ermittelt, wobei das Prinzip der do

  ID: SELF-082  |  ROUGE-L: 0.0207
  Reference: Verschärfungen beim Selbstanzeige-Recht und Einführung der NoVA-Erhöhung.
  Predicted: Das Abgabenänderungsgesetz 2014 (BGBl. I Nr. 114/2014) fungiert als ein umfassendes Omnibusgesetz, welches diverse steuerrechtliche Bestimmungen zur administrativen Vereinfachung, zur Harmonisierung m

  ID: TAX-INTL-007  |  ROUGE-L: 0.0218
  Reference: Nein. Nicht ortsfeste Unterkünfte wie mobile Camping- oder Wohnwagen stellen laut EStR 2000 Rz 21 keinen Wohnsitz dar.
  Predicted: Nach österreichischem Steuerrecht wird ein inländischer Wohnsitz gemäß § 2 Einkommensteuergesetz (EStG

## 8. Summary of Findings

**Model 2 (Gemma 4 E2B, QLoRA fine-tuned) outperforms Model 1 (Gemma 4 31B, zero-shot) across all metrics**, despite having only ~5.1B parameters vs 31B. This demonstrates the effectiveness of domain-specific fine-tuning over raw model scale for specialized legal QA tasks.

**Key error patterns:**
- **Model 1** produces excessively long answers (~4.7x the reference length), which hurts precision-based metrics. It tends to provide general tax explanations rather than concise, paragraph-specific answers.
- **Model 2** matches the reference answer length much better thanks to fine-tuning on a Q&A dataset, but occasionally hallucinates incorrect legal references (BGB instead of KStG) due to the general German law training data mixing civil and tax law concepts.
- **Both models** struggle with precise paragraph citations — they often cite wrong section numbers even when the general legal concept is correct.
- **Repetition** is a minor issue for Model 2 (~4% of answers contain repeated sentences), likely caused by the limited 150-step fine-tuning schedule.